# Limpeza e conversao de tipos - Netflix Titles

In [47]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('data/netflix_titles.csv')
OUTPUT_DIR = Path('outputs')
CLEAN_CSV_PATH = OUTPUT_DIR / 'netflix_titles_clean.csv'
CLEAN_PICKLE_PATH = OUTPUT_DIR / 'netflix_titles_clean.pkl'
MISSING_REPORT_PATH = OUTPUT_DIR / 'missing_values_report.csv'
MIXED_TYPE_REPORT_PATH = OUTPUT_DIR / 'mixed_type_columns_report.csv'
DATE_REPORT_PATH = OUTPUT_DIR / 'date_columns_report.csv'

df = pd.read_csv(DATA_PATH)
df.shape

(8807, 12)

## 1. Analisar colunas de data

In [48]:
df[['date_added', 'release_year']].dtypes

date_added        str
release_year    int64
dtype: object

In [49]:
df[['date_added', 'release_year']].head(10)

,date_added,release_year
0,"September 25, 2021",2020
1,"September 24, 2021",2021
2,"September 24, 2021",2021
3,"September 24, 2021",2021
4,"September 24, 2021",2021
5,"September 24, 2021",2021
6,"September 24, 2021",2021
7,"September 24, 2021",1993
8,"September 24, 2021",2021
9,"September 24, 2021",2021


## 2. Tratar ausentes, tipos mistos e datas

In [50]:
clean = df.copy()

duration_in_rating = clean['rating'].astype('string').str.contains(
    r'\b(?:min|Season|Seasons)\b', case=False, na=False, regex=True
)
duration_missing = duration_in_rating & clean['duration'].isna()
clean.loc[duration_missing, 'duration'] = clean.loc[duration_missing, 'rating']
clean.loc[duration_in_rating, 'rating'] = pd.NA

date_added_text = clean['date_added'].astype('string').str.strip()
clean['date_added_datetime'] = pd.to_datetime(
    date_added_text, format='%B %d, %Y', errors='coerce'
)

columns_to_unknown = ['director', 'cast', 'country', 'date_added', 'rating', 'duration']
clean[columns_to_unknown] = clean[columns_to_unknown].fillna('Unknown')

duration_parts = clean['duration'].str.extract(r'^(?P<value>\d+)\s+(?P<unit>min|Season|Seasons)$')
clean['duration_value'] = duration_parts['value'].astype('int64')
clean['duration_unit'] = (
    duration_parts['unit']
    .str.lower()
    .replace({'min': 'minutes', 'season': 'seasons'})
)

clean[['date_added_datetime', 'release_year', 'duration_value', 'duration_unit']].dtypes

date_added_datetime    datetime64[us]
release_year                    int64
duration_value                  int64
duration_unit                     str
dtype: object

## 3. Validar conversao de datas

In [51]:
date_report = pd.DataFrame([
    {
        'column': 'date_added',
        'original_dtype': str(df['date_added'].dtype),
        'converted_column': 'date_added_datetime',
        'converted_dtype': str(clean['date_added_datetime'].dtype),
        'missing_before': int(df['date_added'].isna().sum()),
        'nat_after_conversion': int(clean['date_added_datetime'].isna().sum()),
        'min_date': clean['date_added_datetime'].min().date().isoformat(),
        'max_date': clean['date_added_datetime'].max().date().isoformat(),
        'treatment': 'Convertida com pd.to_datetime; datas ausentes/invalidas permanecem como NaT.',
    },
    {
        'column': 'release_year',
        'original_dtype': str(df['release_year'].dtype),
        'converted_column': '',
        'converted_dtype': '',
        'missing_before': int(df['release_year'].isna().sum()),
        'nat_after_conversion': '',
        'min_date': '',
        'max_date': '',
        'treatment': 'Mantida como ano numerico, pois nao contem mes e dia para formar data completa.',
    },
])

date_report

,column,original_dtype,converted_column,converted_dtype,missing_before,nat_after_conversion,min_date,max_date,treatment
0,date_added,str,date_added_datetime,datetime64[us],10,10,2008-01-01,2021-09-25,Convertida com pd.to_datetime; datas ausentes/...
1,release_year,int64,,,0,,,,"Mantida como ano numerico, pois nao contem mes..."


## 4. Salvar resultados

In [52]:
missing_treatments = {
    'show_id': 'Sem ausentes; mantida.',
    'type': 'Sem ausentes; mantida.',
    'title': 'Sem ausentes; mantida.',
    'director': "Ausentes preenchidos com 'Unknown'.",
    'cast': "Ausentes preenchidos com 'Unknown'.",
    'country': "Ausentes preenchidos com 'Unknown'.",
    'date_added': 'Ausentes preenchidos com Unknown na coluna original; datetime derivado usa NaT.',
    'release_year': 'Sem ausentes; mantida como ano numerico.',
    'rating': 'Valores de duracao deslocados para duration; ausentes preenchidos com Unknown.',
    'duration': 'Duracoes deslocadas de rating corrigidas; ausentes restantes preenchidos com Unknown.',
    'listed_in': 'Sem ausentes; mantida.',
    'description': 'Sem ausentes; mantida.',
}

missing_report = pd.DataFrame({
    'column': df.columns,
    'missing_before': [int(df[column].isna().sum()) for column in df.columns],
    'missing_before_pct': [round(df[column].isna().mean() * 100, 2) for column in df.columns],
    'missing_after': [int(clean[column].isna().sum()) for column in df.columns],
    'treatment': [missing_treatments[column] for column in df.columns],
})

mixed_type_report = pd.DataFrame([
    {
        'column': 'duration',
        'issue': "Valores combinavam numero e unidade no mesmo texto, como '90 min' e '2 Seasons'.",
        'treatment': 'Criadas duration_value como numero inteiro e duration_unit como unidade padronizada.',
        'new_columns': 'duration_value, duration_unit',
    }
])

OUTPUT_DIR.mkdir(exist_ok=True)
clean.to_csv(CLEAN_CSV_PATH, index=False)
clean.to_pickle(CLEAN_PICKLE_PATH)
missing_report.to_csv(MISSING_REPORT_PATH, index=False)
mixed_type_report.to_csv(MIXED_TYPE_REPORT_PATH, index=False)
date_report.to_csv(DATE_REPORT_PATH, index=False)

clean.shape

(8807, 15)